# Banking Dataset EDA

This notebook loads the CSV files in the current folder, summarizes their structure, checks the main relationships, and sketches a Postgres schema.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_DIR = Path('.')
FILES = {
    'customers': 'customers.csv',
    'accounts': 'accounts.csv',
    'cards': 'cards.csv',
    'loans': 'loans.csv',
    'branches': 'branches.csv',
    'merchants': 'merchants.csv',
}

dfs = {name: pd.read_csv(DATA_DIR / filename) for name, filename in FILES.items()}

for name, df in dfs.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

customers: 50,000 rows x 7 columns
accounts: 75,000 rows x 5 columns
cards: 100,000 rows x 4 columns
loans: 30,000 rows x 5 columns
branches: 500 rows x 3 columns
merchants: 5,000 rows x 3 columns


In [2]:
for name, df in dfs.items():
    print(f"\n=== {name.upper()} ===")
    display(df.head(3))


=== CUSTOMERS ===


,customer_id,first_name,last_name,email,city,credit_score,created_at
0,CUSXAJI0Y6DPBHS,Kevin,Young,brauncameron@example.net,North Williamville,327,2025-04-17
1,CUSHXTHV3A3ZMF8,Jason,Clements,toddwilliam@example.net,Martinezside,644,2020-02-23
2,CUSDD4V30T9NT3W,Randy,Thompson,trevoranderson@example.org,Gallowayfurt,670,2025-06-22



=== ACCOUNTS ===


,account_id,customer_id,account_type,balance_usd,open_date
0,ACCOD48PUCAEHKH,CUSAEOACKBH8CK6,Checking,182700.46,2021-03-27
1,ACCU2DMTSOYD5R6,CUSKI0ABUBGCLFD,Savings,57886.71,2019-10-21
2,ACCXFI6TF26FL9B,CUSWXG0ESCUG77M,Business,68774.44,2019-02-15



=== CARDS ===


...

,loan_id,customer_id,loan_amount,interest_rate,start_date
0,LONVIJLFE0FQERD,CUS0VWWKGL3C2ET,7939.97,8.16,2024-05-05
1,LONHREM3GBFRZCN,CUSK3SW6KH12S96,176269.07,3.84,2022-12-12
2,LONRVF09SWTR92A,CUS7D2AZQQGCDHI,193167.37,10.97,2024-06-01



=== BRANCHES ===


,branch_id,branch_name,manager_name
0,BRNJITQ446NVY1P,Davidberg Branch,Sergio Parker
1,BRNVXROZ3RRD54H,Sanderschester Branch,James Wilson
2,BRN6IUCH832B5CG,West Aprilchester Branch,Benjamin Smith



=== MERCHANTS ===


,merchant_id,merchant_name,city
0,MERU9JCGLS9MNLI,"Williams, Moore and Campbell",West Bradley.
1,MER43QNBEL8JWEW,Wolf LLC,New Heather
2,MER3N5P7AWGR7U5,Smith and Sons,Lake Kevinbury


In [3]:
def summarize_dataframe(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'nulls': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'unique_values': df.nunique(dropna=True),
    })

for name, df in dfs.items():
    print(f"\n=== {name.upper()} SUMMARY ===")
    display(summarize_dataframe(df))


=== CUSTOMERS SUMMARY ===


,dtype,non_null,nulls,null_pct,unique_values
customer_id,object,50000,0,0.0,50000
first_name,object,50000,0,0.0,690
last_name,object,50000,0,0.0,1000
email,object,50000,0,0.0,45773
city,object,50000,0,0.0,25169
credit_score,int64,50000,0,0.0,551
created_at,object,50000,0,0.0,2557



=== ACCOUNTS SUMMARY ===


,dtype,non_null,nulls,null_pct,unique_values
account_id,object,75000,0,0.0,75000
customer_id,object,75000,0,0.0,38838
account_type,object,75000,0,0.0,3
balance_usd,float64,75000,0,0.0,74846
open_date,object,75000,0,0.0,2557



=== CARDS SUMMARY ===


...

,dtype,non_null,nulls,null_pct,unique_values
loan_id,object,30000,0,0.0,30000
customer_id,object,30000,0,0.0,22586
loan_amount,float64,30000,0,0.0,29985
interest_rate,float64,30000,0,0.0,1301
start_date,object,30000,0,0.0,2557



=== BRANCHES SUMMARY ===


,dtype,non_null,nulls,null_pct,unique_values
branch_id,object,500,0,0.0,500
branch_name,object,500,0,0.0,494
manager_name,object,500,0,0.0,500



=== MERCHANTS SUMMARY ===


,dtype,non_null,nulls,null_pct,unique_values
merchant_id,object,5000,0,0.0,5000
merchant_name,object,5000,0,0.0,4557
city,object,5000,0,0.0,4313


In [4]:
display(dfs['accounts']['account_type'].value_counts().rename('account_count').to_frame())
display(dfs['cards']['card_type'].value_counts().rename('card_count').to_frame())

numeric_describe = {
    'customers.credit_score': dfs['customers']['credit_score'].describe(),
    'accounts.balance_usd': dfs['accounts']['balance_usd'].describe(),
    'loans.loan_amount': dfs['loans']['loan_amount'].describe(),
    'loans.interest_rate': dfs['loans']['interest_rate'].describe(),
}

for label, series in numeric_describe.items():
    print(f"\n=== {label} ===")
    display(series.to_frame(name='value'))

,account_count
account_type,
Checking,25090
Savings,24962
Business,24948


,card_count
card_type,
Debit,50281
Credit,49719



=== customers.credit_score ===


,value
count,50000.000000
mean,574.490520
std,158.692145
min,300.000000
25%,437.000000
50%,575.000000
75%,712.000000
max,850.000000



=== accounts.balance_usd ===


,value
count,75000.000000
mean,100181.721881
std,57637.062728
min,13.200000
25%,50406.112500
50%,100316.825000
75%,149973.740000
max,199994.580000



=== loans.loan_amount ===


,value
count,30000.000000
mean,150089.016603
std,86370.873092
min,1010.160000
25%,75500.137500
50%,150571.300000
75%,224719.092500
max,299975.470000



=== loans.interest_rate ===


,value
count,30000.000000
mean,8.507418
std,3.752818
min,2.000000
25%,5.270000
50%,8.510000
75%,11.770000
max,15.000000


In [5]:
relationship_summary = pd.DataFrame([
    {
        'relationship': 'accounts -> customers',
        'child_rows': len(dfs['accounts']),
        'parent_rows': len(dfs['customers']),
        'orphan_child_rows': (~dfs['accounts']['customer_id'].isin(dfs['customers']['customer_id'])).sum(),
        'parents_with_children': dfs['customers']['customer_id'].isin(dfs['accounts']['customer_id']).sum(),
    },
    {
        'relationship': 'cards -> accounts',
        'child_rows': len(dfs['cards']),
        'parent_rows': len(dfs['accounts']),
        'orphan_child_rows': (~dfs['cards']['account_id'].isin(dfs['accounts']['account_id'])).sum(),
        'parents_with_children': dfs['accounts']['account_id'].isin(dfs['cards']['account_id']).sum(),
    },
    {
        'relationship': 'loans -> customers',
        'child_rows': len(dfs['loans']),
        'parent_rows': len(dfs['customers']),
        'orphan_child_rows': (~dfs['loans']['customer_id'].isin(dfs['customers']['customer_id'])).sum(),
        'parents_with_children': dfs['customers']['customer_id'].isin(dfs['loans']['customer_id']).sum(),
    },
])

display(relationship_summary)

,relationship,child_rows,parent_rows,orphan_child_rows,parents_with_children
0,accounts -> customers,75000,50000,0,38838
1,cards -> accounts,100000,75000,0,55198
2,loans -> customers,30000,50000,0,22586


In [6]:
multiplicity = {
    'accounts_per_customer': dfs['accounts'].groupby('customer_id').size().describe(),
    'cards_per_account': dfs['cards'].groupby('account_id').size().describe(),
    'loans_per_customer': dfs['loans'].groupby('customer_id').size().describe(),
}

for label, series in multiplicity.items():
    print(f"\n=== {label} ===")
    display(series.to_frame(name='value'))


=== accounts_per_customer ===


,value
count,38838.000000
mean,1.931098
std,1.039524
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,8.000000



=== cards_per_account ===


,value
count,55198.000000
mean,1.811660
std,0.969917
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,8.000000



=== loans_per_customer ===


,value
count,22586.000000
mean,1.328256
std,0.601242
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,6.000000


In [7]:
quality_checks = pd.DataFrame([
    {'check': 'duplicate customer emails', 'value': int(dfs['customers']['email'].duplicated().sum())},
    {'check': 'duplicate branch names', 'value': int(dfs['branches']['branch_name'].duplicated().sum())},
    {'check': 'duplicate merchant names', 'value': int(dfs['merchants']['merchant_name'].duplicated().sum())},
    {'check': 'negative account balances', 'value': int((dfs['accounts']['balance_usd'] < 0).sum())},
    {'check': 'negative loan amounts', 'value': int((dfs['loans']['loan_amount'] < 0).sum())},
    {'check': 'credit scores outside 300-850', 'value': int(((dfs['customers']['credit_score'] < 300) | (dfs['customers']['credit_score'] > 850)).sum())},
    {'check': 'interest rates outside 0-100', 'value': int(((dfs['loans']['interest_rate'] < 0) | (dfs['loans']['interest_rate'] > 100)).sum())},
])

display(quality_checks)

,check,value
0,duplicate customer emails,4227
1,duplicate branch names,6
2,duplicate merchant names,443
3,negative account balances,0
4,negative loan amounts,0
5,credit scores outside 300-850,0
6,interest rates outside 0-100,0


In [8]:
date_ranges = pd.DataFrame([
    {'field': 'customers.created_at', 'min': dfs['customers']['created_at'].min(), 'max': dfs['customers']['created_at'].max()},
    {'field': 'accounts.open_date', 'min': dfs['accounts']['open_date'].min(), 'max': dfs['accounts']['open_date'].max()},
    {'field': 'cards.expiration_date', 'min': dfs['cards']['expiration_date'].min(), 'max': dfs['cards']['expiration_date'].max()},
    {'field': 'loans.start_date', 'min': dfs['loans']['start_date'].min(), 'max': dfs['loans']['start_date'].max()},
])

coverage = pd.DataFrame([
    {'metric': 'customers with accounts', 'value': int(dfs['customers']['customer_id'].isin(dfs['accounts']['customer_id']).sum())},
    {'metric': 'customers with loans', 'value': int(dfs['customers']['customer_id'].isin(dfs['loans']['customer_id']).sum())},
    {'metric': 'accounts with cards', 'value': int(dfs['accounts']['account_id'].isin(dfs['cards']['account_id']).sum())},
])

display(date_ranges)
display(coverage)

,field,min,max
0,customers.created_at,2019-01-01,2025-12-31
1,accounts.open_date,2019-01-01,2025-12-31
2,cards.expiration_date,2025-01-01,2032-12-31
3,loans.start_date,2019-01-01,2025-12-31


,metric,value
0,customers with accounts,38838
1,customers with loans,22586
2,accounts with cards,55198


## Suggested Postgres Model

The natural core model in this data is:

- `customers`
- `accounts` referencing `customers`
- `cards` referencing `accounts`
- `loans` referencing `customers`
- `branches` and `merchants` as standalone dimension tables unless more relationship columns are added later

Suggested types:

- IDs: `text` primary keys
- money-like values: `numeric(14,2)`
- dates: `date`
- score/rate columns with `check` constraints

In [9]:
schema_sql = '''
create table customers (
    customer_id text primary key,
    first_name text not null,
    last_name text not null,
    email text not null,
    city text not null,
    credit_score integer not null check (credit_score between 300 and 850),
    created_at date not null
);

create table accounts (
    account_id text primary key,
    customer_id text not null references customers(customer_id),
    account_type text not null check (account_type in ('Checking', 'Savings', 'Business')),
    balance_usd numeric(14,2) not null check (balance_usd >= 0),
    open_date date not null
);

create table cards (
    card_id text primary key,
    account_id text not null references accounts(account_id),
    card_type text not null check (card_type in ('Debit', 'Credit')),
    expiration_date date not null
);

create table loans (
    loan_id text primary key,
    customer_id text not null references customers(customer_id),
    loan_amount numeric(14,2) not null check (loan_amount >= 0),
    interest_rate numeric(5,2) not null check (interest_rate >= 0 and interest_rate <= 100),
    start_date date not null
);

create table branches (
    branch_id text primary key,
    branch_name text not null,
    manager_name text not null
);

create table merchants (
    merchant_id text primary key,
    merchant_name text not null,
    city text not null
);
'''

print(schema_sql)


create table customers (
    customer_id text primary key,
    first_name text not null,
    last_name text not null,
    email text not null,
    city text not null,
    credit_score integer not null check (credit_score between 300 and 850),
    created_at date not null
);

create table accounts (
    account_id text primary key,
    customer_id text not null references customers(customer_id),
    account_type text not null check (account_type in ('Checking', 'Savings', 'Business')),
    balance_usd numeric(14,2) not null check (balance_usd >= 0),
    open_date date not null
);

create table cards (
    card_id text primary key,
    account_id text not null references accounts(account_id),
    card_type text not null check (card_type in ('Debit', 'Credit')),
    expiration_date date not null
);

create table loans (
    loan_id text primary key,
    customer_id text not null references customers(customer_id),
    loan_amount numeric(14,2) not null check (loan_amount >= 0),
    intere